# GateGuard Classifier — Baseline Training (Colab)

EfficientNet-B0 영상 분류 베이스라인을 Colab GPU(T4) 에서 학습한다.

**전제**: Google Drive 에 데이터셋이 다음 구조로 있다고 가정.
```
<DATA_ROOT>/
  jump/        *.mp4
  crawling/    *.mp4
  tailgating/  *.mp4
  unpaid/      *.mp4
```
구조가 다르면 4번 셀에서 정리 한 번 해주거나 코드를 맞게 수정한다.

**런타임**: 상단 메뉴 → Runtime → Change runtime type → **GPU(T4)** 선택.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. repo clone + 의존성
!git clone --depth 1 https://github.com/CHOSOOGEUN/GateGuard.git
%cd GateGuard
!pip install -q -r ai/classifier/requirements.txt

In [ ]:
# 3. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. 데이터 경로 설정 + 검증
#  ↓ 본인 드라이브 구조에 맞게 한 줄만 수정.
#    예) '/content/drive/MyDrive/GateGuard_dataset'
#    공유 드라이브면 '/content/drive/Shareddrives/...'
DATA_ROOT = '/content/drive/MyDrive/GateGuard_dataset'

import os, glob
assert os.path.isdir(DATA_ROOT), f'경로 없음: {DATA_ROOT}'
for cls in ('jump', 'crawling', 'tailgating', 'unpaid'):
    p = os.path.join(DATA_ROOT, cls)
    n = len(glob.glob(os.path.join(p, '*.mp4'))) if os.path.isdir(p) else 0
    flag = 'OK ' if n > 0 else '!! '
    print(f'{flag}{cls:12s} {n:3d} clips  ({p})')

os.environ['DATA_ROOT'] = DATA_ROOT

In [ ]:
# 5. 학습 실행
#  - epoch 50, batch 8, AdamW + CE(class_weight balanced), early stop patience 7
#  - val_loss 기준 best 저장 → ai/classifier/runs/baseline/best_model.pth
#  - 종료 후 confusion matrix + classification report 출력
!python -m ai.classifier.train \
    --data-root "$DATA_ROOT" \
    --output-dir ai/classifier/runs/baseline \
    --epochs 50 --batch-size 8

In [ ]:
# 6. (선택) 학습 결과를 드라이브로 백업
import datetime, shutil, os
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = f'/content/drive/MyDrive/GateGuard_runs/baseline_{stamp}'
os.makedirs(os.path.dirname(dst), exist_ok=True)
shutil.copytree('ai/classifier/runs/baseline', dst)
print(f'saved: {dst}')

In [ ]:
# 7. (선택) 단일 영상 추론 데모
#    SAMPLE_VIDEO 에 본인 클립 하나 경로 넣기
SAMPLE_VIDEO = '/content/drive/MyDrive/GateGuard_dataset/jump/clip001.mp4'
!python -m ai.classifier.inference_video \
    --checkpoint ai/classifier/runs/baseline/best_model.pth \
    --video "$SAMPLE_VIDEO"